# Notebook 1: Build BFCL Subset

This notebook constructs a reproducible subset of the Berkeley Function Calling Leaderboard (BFCL) dataset for hardware efficiency experiments.

The output is a fixed benchmark subset saved to `data/bfcl_subset.json`.

This subset will be reused across all accelerator runs to ensure that T4, L4, RTX 6000 Ada, and later TPU experiments use the same tasks.

In [32]:
import json
import random
from pathlib import Path
from collections import Counter

import pandas as pd
from huggingface_hub import hf_hub_download

In [33]:
DATA_DIR = Path("../data")
DATA_DIR.mkdir(exist_ok=True)

SUBSET_PATH = DATA_DIR / "bfcl_subset.json"
SUBSET_METADATA_PATH = DATA_DIR / "bfcl_subset_metadata.json"

RANDOM_SEED = 188
TARGET_N = 100
random.seed(RANDOM_SEED)

## Load BFCL Dataset

We first attempt to load the BFCL dataset from Hugging Face. If dataset structure differs across versions, we inspect the available splits and columns before selecting tasks.

In [34]:
REPO_ID   = "gorilla-llm/Berkeley-Function-Calling-Leaderboard"
RANDOM_SEED = 188
TARGET_N    = 100
random.seed(RANDOM_SEED)

DATA_DIR = Path("../data")
DATA_DIR.mkdir(exist_ok=True)

# ── Correct v3 filenames ──────────────────────────────────────────────────────
# We use the AST (non-exec) variants for simple/multiple/parallel since those
# pair cleanly with possible_answer/. The exec variants contain ground-truth
# inline, so they can be loaded without a separate answer file.
FILE_PAIRS = [
    ("BFCL_v3_simple.json",            "possible_answer/BFCL_v3_simple.json"),
    ("BFCL_v3_multiple.json",          "possible_answer/BFCL_v3_multiple.json"),
    ("BFCL_v3_parallel.json",          "possible_answer/BFCL_v3_parallel.json"),
    ("BFCL_v3_parallel_multiple.json", "possible_answer/BFCL_v3_parallel_multiple.json"),
]

def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def download(filename):
    local_path = hf_hub_download(
        repo_id=REPO_ID,
        filename=filename,
        repo_type="dataset"
    )
    return load_jsonl(local_path)

# ── Load + join questions with answers ───────────────────────────────────────
all_rows = []

for question_file, answer_file in FILE_PAIRS:
    print(f"Loading {question_file}...")
    questions = download(question_file)
    answers   = download(answer_file)

    # answers are indexed by id
    answer_map = {a["id"]: a for a in answers}
    category   = question_file.replace("BFCL_v3_", "").replace(".json", "")

    for q in questions:
        task_id = q.get("id")
        answer  = answer_map.get(task_id, {})

        row = {
            "task_id":            task_id,
            "category":           category,
            "prompt":             q.get("question"),
            "functions_or_tools": q.get("function"),
            "gold_answer":        answer.get("ground_truth"),
            "execution_result_type": answer.get("execution_result_type"),
        }

        if row["prompt"] and row["functions_or_tools"] and row["gold_answer"]:
            all_rows.append(row)

print(f"\nUsable rows after join: {len(all_rows)}")
print("By category:", Counter(r["category"] for r in all_rows))

# ── Stratified sample (25 per category) ─────────────────────────────────────
# More informative than pure random for a paper
by_category = {}
for row in all_rows:
    by_category.setdefault(row["category"], []).append(row)

n_per_category = TARGET_N // len(by_category)
subset = []
for cat, rows in by_category.items():
    sample = random.sample(rows, min(n_per_category, len(rows)))
    subset.extend(sample)
    print(f"  {cat}: {len(sample)} tasks")

print(f"\nTotal subset: {len(subset)} tasks")

# ── Save ─────────────────────────────────────────────────────────────────────
SUBSET_PATH   = DATA_DIR / "bfcl_subset.json"
METADATA_PATH = DATA_DIR / "bfcl_subset_metadata.json"

with open(SUBSET_PATH, "w") as f:
    json.dump(subset, f, indent=2)

with open(METADATA_PATH, "w") as f:
    json.dump({
        "dataset_version": "BFCL_v3",
        "random_seed":     RANDOM_SEED,
        "target_n":        TARGET_N,
        "actual_n":        len(subset),
        "n_per_category":  n_per_category,
        "categories":      dict(Counter(r["category"] for r in subset)),
        "notes":           "Stratified BFCL v3 subset for hardware-aware inference experiments."
    }, f, indent=2)

print(f"Saved to {SUBSET_PATH}")

Loading BFCL_v3_simple.json...
Loading BFCL_v3_multiple.json...
Loading BFCL_v3_parallel.json...
Loading BFCL_v3_parallel_multiple.json...

Usable rows after join: 1000
By category: Counter({'simple': 400, 'multiple': 200, 'parallel': 200, 'parallel_multiple': 200})
  simple: 25 tasks
  multiple: 25 tasks
  parallel: 25 tasks
  parallel_multiple: 25 tasks

Total subset: 100 tasks
Saved to ../data/bfcl_subset.json


In [35]:
# ── Verify saved subset ───────────────────────────────────────────────────────
with open(SUBSET_PATH, "r", encoding="utf-8") as f:
    loaded_subset = json.load(f)

print("Reloaded tasks:", len(loaded_subset))
print("Categories:", Counter(r["category"] for r in loaded_subset))

# Spot-check first row
first = loaded_subset[0]
print("\ntask_id:", first["task_id"])
print("category:", first["category"])
print("prompt preview:", str(first["prompt"])[:200])
print("tools count:", len(first["functions_or_tools"]) if isinstance(first["functions_or_tools"], list) else 1)
print("gold_answer:", first["gold_answer"])

Reloaded tasks: 100
Categories: Counter({'simple': 25, 'multiple': 25, 'parallel': 25, 'parallel_multiple': 25})

task_id: simple_361
category: simple
prompt preview: [[{'role': 'user', 'content': 'Find Italian restaurants near New York city that serves gluten-free options.'}]]
tools count: 1
gold_answer: [{'restaurant_finder': {'city': ['New York City', 'New York City, NY', 'NYC', 'New York'], 'cuisine': ['Italian'], 'diet': ['Gluten-free']}}]


## Notes

This subset is now fixed for the first iteration. Later notebooks should not resample BFCL. They should only load `data/bfcl_subset.json`.